In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
from Clean_Data import get_cleaned_vanguard_data
from scipy.stats import chi2_contingency

In [ ]:
df = get_cleaned_vanguard_data(True)
df

In [ ]:
def check_succesful_path(x):
    list=['perfect_path','successful_with_repeats']
    return x in list

client_level = df.groupby('client_id').agg({
    'variation': 'first',
    'age': 'first',
    'path_category': lambda x: check_succesful_path(x.iloc[0])
}).rename(columns={'path_category': 'is_completed'}).reset_index()

client_level = client_level.dropna(subset=['variation', 'age'])

bins = [0, 25, 40, 60, 120]
labels = ['Gen Z (Under 25)', 'Millennials (26-40)', 'Gen X (41-60)', 'Boomers/Retirees (60+)']
client_level['age_group'] = pd.cut(client_level['age'], bins=bins, labels=labels)

client_level


In [ ]:
client_level['age_group'].cat.categories

In [ ]:
for age_cat in client_level['age_group'].cat.categories:

    print(f"For Age Category: {age_cat}")
    group_data = client_level[client_level['age_group'] == age_cat]
    variation_test_data = group_data[group_data['variation'] == 'Test']
    variation_test_success = variation_test_data['is_completed'].sum()
    variation_test_fail = len(variation_test_data) - variation_test_success
    variation_control_data = group_data[group_data['variation'] == 'Control']
    variation_control_success = variation_control_data['is_completed'].sum()
    variation_control_fail = len(variation_control_data) - variation_control_success
    
    if len(variation_test_data) == 0 or len(variation_control_data) == 0:
        print("  Not enough data for this category.\n")
        continue
    
    observed = np.array([
        [variation_test_success, variation_test_fail],
        [variation_control_success, variation_control_fail]
    ])
    
    chi2_stat, p_val, dof, expected = chi2_contingency(observed)

    test_rate = variation_test_success / len(variation_test_data)
    control_rate = variation_control_success / len(variation_control_data)
    
    print(f"  Test Completion Rate:    {test_rate:.2%}")
    print(f"  Control Completion Rate: {control_rate:.2%}")
    print(f"  P-value: {p_val:.4e}")
    
    if p_val < 0.05:
        if test_rate > control_rate:
            print(f"Conclusion: The Test UI performed SIGNIFICANTLY BETTER for {age_cat}.\n")
        else:
            print(f"Conclusion: The Test UI performed SIGNIFICANTLY WORSE for {age_cat}.\n")
    else:
        print(f"➖ Conclusion: NO significant difference between Test and Control for {age_cat}.\n")


In [ ]:
test_group = client_level[client_level['variation'] == 'Test']

age_completed = test_group[test_group['is_completed'] == True]['age']
age_failed = test_group[test_group['is_completed'] == False]['age']

t_stat, p_val = stats.ttest_ind(age_completed, age_failed)

print(f"Average Age (Completed): {age_completed.mean():.2f}")
print(f"Average Age (Failed): {age_failed.mean():.2f}")
print("-" * 30)
print(f"T-statistic: {t_stat:.4f}")
print(f"P-value: {p_val:.4e}")
print("-" * 30)

if p_val < 0.05:
    print("Conclusion: Reject H0.")
    print("There is a statistically significant difference in age between those who complete the new UI and those who drop off.")
else:
    print("Conclusion: Fail to reject H0.")
    print("There is NO significant difference in age between those who complete the new UI and those who drop off.")


The new Test UI is highly successful and should be rolled out to all users. It significantly improves completion rates for every age group, including our senior demographics. 
However, as older users are still slightly more likely to drop off than younger users, future iterations of the UI could focus on accessibility or simplified flows to further assist the 50+ demographic."